# Elicitation Robustness Check

**Question:** Could the declarative-evaluative gap be a prompting artifact?

The original evaluative prompts (Experiment 2) explicitly contain the word
"accessible" or "accessibility" — e.g., *"The following code is not accessible
because it doesn't have what?"* A reviewer could argue the model is
pattern-matching on that keyword rather than demonstrating applied knowledge.

This notebook tests the same underlying capability using structural HTML
completion prompts that contain **no accessibility-related language**. The model
either completes the HTML correctly (demonstrating applied knowledge) or it
doesn't. Following the elicitation robustness methodology used in recent
mechanistic interpretability work (cf. prompt paraphrasing in circuit stability
analysis; multiple elicitation strategies in introspection replication studies),
we vary how we ask while holding what we test constant.

**Design:**
- 2 elicitation strategies: few-shot structural completion, bare completion
- 3 accessibility concepts: alt text, closed captions (track element), page title
- 1 control concept: script src (non-accessibility HTML pattern)
- 2 models: Pythia 2.8B (emergence threshold), Pythia 1B (predicted dead zone)
- Greedy decoding, temperature 0

**What we expect:**
- Pythia 2.8B: completes control patterns correctly; fails or partially fails
  accessibility patterns — the gap persists even without keyword priming
- Pythia 1B: fails both — confirming general capability boundary, not domain-specific

## Setup

In [4]:
import torch
from transformer_lens import HookedTransformer

In [5]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"Using device: {device}")

Using device: mps


## Prompts

Two elicitation strategies, no accessibility language in any prompt.

**Few-shot structural:** Establish a pattern with 2 correct examples, then
present an incomplete third. Tests whether the model continues the pattern.

**Bare completion:** Present the incomplete HTML with no prior context.
Tests whether the model produces the correct attribute unprompted.

In [11]:
# -- Alt Text: img -> alt relationship --

alt_fewshot = """
    <img src="icons/home.png" alt="Home">
    <img src="icons/mail.png" alt="Inbox">
    <img src="photo.jpg" """

alt_bare = '<img src="photo.jpg"'

# -- Closed Captions: video -> track relationship --

captions_fewshot = """<video src="intro.mp4"><track kind="captions" src="intro.vtt"></video>
<video src="demo.mp4"><track kind="captions" src="demo.vtt"></video>
<video src="lecture.mp4">"""

captions_bare = '<video src="lecture.mp4">'

# -- Page Title: head -> title relationship --

title_fewshot = """<html><head><title>About Us</title></head>
<html><head><title>Contact</title></head>
<html><head>"""

title_bare = '<html><head><meta charset="utf-8">'

# -- Control: script -> src relationship (non-accessibility) --

control_fewshot = """
<script src="utils.js"></script>
<script src="main.js"></script>
<script src=" """

control_bare = '<script'

# All prompts for iteration
prompts = [
    ("alt text",         "fewshot", alt_fewshot),
    ("alt text",         "bare",    alt_bare),
    ("closed captions",  "fewshot", captions_fewshot),
    ("closed captions",  "bare",    captions_bare),
    ("page title",       "fewshot", title_fewshot),
    ("page title",       "bare",    title_bare),
    ("script (control)", "fewshot", control_fewshot),
    ("script (control)", "bare",    control_bare),
]

MAX_TOKENS = 20

print(f"{len(prompts)} prompts, {MAX_TOKENS} max tokens")

8 prompts, 20 max tokens


## Pythia 2.8B

*Emergence threshold — declarative knowledge confirmed in original experiments.
Question: does applied knowledge show up in structural HTML completion?*

### Load Model

In [12]:
model = HookedTransformer.from_pretrained("pythia-2.8b", device=device)
print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-2.8b into HookedTransformer
Layers: 32
Heads: 32
Params: 2774.9M


### Run Prompts

In [13]:
for concept, strategy, prompt in prompts:
    output = model.generate(prompt, max_new_tokens=MAX_TOKENS, temperature=0, verbose=False)
    completion = output[len(prompt):]
    print(f"{concept:20} [{strategy:7}] -> {completion}")
    print()

alt text             [fewshot] -> 
alt="Photo">
    <img src="icons/search.png" alt="Search

alt text             [bare   ] ->  alt="photo" width="300" height="300" />

<p>
  

closed captions      [fewshot] -> 
<track kind="captions" src="lecture.vtt"></video>
<

closed captions      [bare   ] -> 
  <source src="lecture.webm" type="video/webm" />

page title           [fewshot] -> 
<meta http-equiv="Content-Type" content="text/html; charset=iso

page title           [bare   ] -> 
<title>CSS Test Reference</title>
<link rel="author" title="L

script (control)     [fewshot] -> 
<script src="
<script src="
<script src="
<script src="

script (control)     [bare   ] -> >
  import {
    useTranslation,
    useForm,
    useField,



### Observations

*Record observations here after running. Key questions:*
- *Does the model add `alt` in the few-shot pattern? In bare?*
- *Does it add `<track>` for video? `<title>` for head?*
- *Does it correctly complete `<script src=`?*
- *Where does it succeed vs. fail relative to the control?*

### Delete Model & Clear Cache

In [ ]:
del model
if device == "cuda":
    torch.cuda.empty_cache()
elif device == "mps":
    torch.mps.empty_cache()
print("Model deleted, cache cleared.")

## Pythia 1B

*Predicted dead zone — fails evaluative reasoning across all domains in
original experiments. If 1B also fails these structural completions, the
failure is general capability, not domain-specific.*

### Load Model

In [ ]:
model = HookedTransformer.from_pretrained("pythia-1b", device=device)
print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

### Run Prompts

In [ ]:
for concept, strategy, prompt in prompts:
    output = model.generate(prompt, max_new_tokens=MAX_TOKENS, temperature=0, verbose=False)
    completion = output[len(prompt):]
    print(f"{concept:20} [{strategy:7}] -> {completion}")
    print()

### Observations

*Record observations here after running.*

### Delete Model & Clear Cache

In [ ]:
del model
if device == "cuda":
    torch.cuda.empty_cache()
elif device == "mps":
    torch.mps.empty_cache()
print("Model deleted, cache cleared.")

## Summary

*Fill in after running both models. Key narrative:*

- *If 2.8B completes control correctly but fails accessibility -> gap confirmed,
  not a prompting artifact*
- *If 2.8B fails both -> general HTML completion failure (different finding)*
- *If 1B fails everything -> confirms general capability boundary*
- *Compare few-shot vs bare: does structural priming help? If so, knowledge
  may be latent but require activation — worth noting*